In [16]:
%pip install httpcore==0.16.3


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import os
import deepl
from dotenv import load_dotenv
from deep_translator import GoogleTranslator
import time
from deep_translator.exceptions import TooManyRequests

In [18]:
# Load environment variables from .env file
load_dotenv()
DEEPL_API_KEY = os.getenv("DEEPL_API_KEY") 
auth_key = DEEPL_API_KEY # replace with your key
deepl_client = deepl.DeepLClient(auth_key)


In [19]:
# === Input file path ===
input_path = r"C:\Users\SakshiMehta\OneDrive - Denison Consulting\Text Analytics data\Creekstone 2026\Creekstone 2026.xlsx"
company_name = "Creekstone Farms"
EU_client = False

In [20]:
df = pd.read_excel(input_path, keep_default_na=False) 

#  Get folder path of input file 
output_folder = os.path.dirname(input_path)
print(output_folder)

C:\Users\SakshiMehta\OneDrive - Denison Consulting\Text Analytics data\Creekstone 2026


In [ ]:
import json


def translate_deepl_safely(
    texts,
    deepl_client,
    target_lang="EN-US",
    max_batch_bytes=95 * 1024,
    max_batch_items=40
):
    """
    Translate texts in batches that remain below DeepL's
    128 KiB total request-size limit.

    Returns translations in the same order as the input.
    """

    all_translations = []
    current_batch = []

    def request_size(batch):
        """
        Estimate the size of the JSON request body in UTF-8 bytes.
        """
        payload = {
            "text": batch,
            "target_lang": target_lang
        }

        return len(
            json.dumps(
                payload,
                ensure_ascii=False
            ).encode("utf-8")
        )

    def translate_batch(batch):
        """
        Send one batch to DeepL and return its translated strings.
        """
        results = deepl_client.translate_text(
            batch,
            target_lang=target_lang
        )

        # DeepL returns a single object for a single string
        # and a list for a list of strings.
        if not isinstance(results, list):
            results = [results]

        return [result.text for result in results]

    for text in texts:
        text = str(text).strip()
        proposed_batch = current_batch + [text]

        batch_too_large = (
            request_size(proposed_batch) > max_batch_bytes
        )

        too_many_items = (
            len(proposed_batch) > max_batch_items
        )

        if current_batch and (batch_too_large or too_many_items):
            all_translations.extend(
                translate_batch(current_batch)
            )

            current_batch = [text]

            print(
                f"Translated {len(all_translations)} "
                f"of {len(texts)} responses"
            )

        else:
            current_batch = proposed_batch

        # Catch an unusually large individual response
        if (
            len(current_batch) == 1
            and request_size(current_batch) > max_batch_bytes
        ):
            raise ValueError(
                "One individual response is too large to send "
                "to DeepL safely. Response begins with: "
                f"{text[:100]!r}"
            )

    # Translate the final incomplete batch
    if current_batch:
        all_translations.extend(
            translate_batch(current_batch)
        )

    return all_translations

In [ ]:
def translate_google_safely(
    texts,
    translator,
    batch_size=4,
    pause_seconds=1.2,
    max_retries=5
):
    """
    Translate text using GoogleTranslator in controlled batches.

    Parameters
    ----------
    texts : list[str]
        Text values to translate.

    translator : GoogleTranslator
        Initialized GoogleTranslator object.

    batch_size : int
        Number of translations attempted before pausing.
        Four is used to stay below the reported five-request-per-second limit.

    pause_seconds : float
        Number of seconds to wait between batches.

    max_retries : int
        Maximum number of retries when Google returns a rate-limit error.

    Returns
    -------
    list[str]
        Translated text in the same order as the input.
    """

    all_translations = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        for attempt in range(max_retries):
            try:
                batch_translations = translator.translate_batch(batch)

                # Make sure the result is always treated as a list
                if isinstance(batch_translations, str):
                    batch_translations = [batch_translations]

                all_translations.extend(batch_translations)
                break

            except TooManyRequests:
                if attempt == max_retries - 1:
                    raise

                # Wait longer after each failed attempt:
                # 5, 10, 20, 40 seconds, etc.
                wait_seconds = 5 * (2 ** attempt)

                print(
                    f"Google rate limit reached. "
                    f"Waiting {wait_seconds} seconds before retrying..."
                )

                time.sleep(wait_seconds)

        # Pause before sending the next batch
        time.sleep(pause_seconds)

    return all_translations

In [21]:
def preprocess_data(df, cols_to_drop=("Oe4",)):
    df.columns = df.iloc[0] # Set the first row as the header
    df = df[2:] # Skip the first two rows which are not part of the data
    df.reset_index(drop=True, inplace=True)


    df = df.iloc[:, 4:]  # Drop the first 4 column
    # Keep column "Status"
    status_idx = df.columns.get_loc("Status")
    # Find the first column starting with "Oe" after "Status"
    after_status = df.columns[status_idx + 1:]
    oe_cols = [col for col in after_status if col.startswith("Oe")]
    if oe_cols:
        first_oe_idx = df.columns.get_loc(oe_cols[0])
        # Drop columns between "Status" and the first "Oe"
        df = df.drop(columns=df.columns[status_idx + 1:first_oe_idx])

        # Drop extra columns if provided
    if cols_to_drop:
        df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

    # Filter rows where Status is "Submitted - Valid"
    df = df[df["Status"] == "Submitted - Valid"].reset_index(drop=True)

    df = df.dropna(axis=1, how='all')
    # Drop columns where all values are either NaN or empty/whitespace strings
    df = df.loc[:, ~df.apply(lambda col: col.astype(str).str.strip().eq('').all())]



    return df

df_cleaned = preprocess_data(df)

In [ ]:
oe_columns = [col for col in df_cleaned.columns if col.startswith("Oe")]
oe_dfs = {}  # dictionary to store separate dataframes


if not EU_client:
    # Initialize translator
    translator = GoogleTranslator(source='auto', target='en')


for oe_col in oe_columns:
    # Keep all non-'oe' columns + the current 'oe' column
    cols_to_keep = [col for col in df_cleaned.columns if not col in oe_columns or col == oe_col]
    
    # Create the filtered DataFrame
    df_part = df_cleaned[cols_to_keep].copy()

    # Identify records containing an actual response
    has_response = (
        df_part[oe_col].notna()
        & df_part[oe_col].astype(str).str.strip().ne("")
    )

    # Remove records where the current response is blank
    df_part = df_part.loc[has_response].copy()


    # Remove unnecessary leading or trailing spaces
    df_part[oe_col] = df_part[oe_col].astype(str).str.strip()



    # Add an index column named "Res_ID" as the first column
    df_part.insert(0, 'Res_ID', range(1, len(df_part) + 1))

    
    # Translate the 'oe_col' column to English
    translated_col = f"{oe_col}_Trans"

    # Convert responses to a list for batch translation
    texts_to_translate = df_part[oe_col].tolist()

    if texts_to_translate:
        if EU_client:
            translations = translate_deepl_safely(
                texts=texts_to_translate,
                deepl_client=deepl_client,
                target_lang="EN-US"
            )

        else:
            translations = translate_google_safely(
                texts=texts_to_translate,
                translator=translator
            )

        df_part[translated_col] = translations

    else:
        # This is mainly a safeguard because blank records
        # have already been removed
        df_part[translated_col] = pd.Series(
            dtype="object",
            index=df_part.index
        )

    # Use the original response if translation is missing or blank
    translation_is_blank = (
        df_part[translated_col].isna()
        | df_part[translated_col]
            .astype(str)
            .str.strip()
            .eq("")
    )

    df_part.loc[translation_is_blank, translated_col] = (
        df_part.loc[translation_is_blank, oe_col]
    )

    # Move the translated column to the third position
    df_part.insert(2, translated_col, df_part.pop(translated_col))

    # Sort by length of translated text (ascending)
    df_part = df_part.sort_values(by=translated_col, key=lambda col: col.str.len(), ascending=True)


    # Save to dictionary
    oe_dfs[oe_col] = df_part
    
    translator_name = "DeepL" if EU_client else "Google"
    output_path = os.path.join(output_folder, f"{company_name}_{oe_col}_{translator_name}_Trans.xlsx"
)
    df_part.to_excel(output_path, index=False)
    
    # replace all the blank values in the translated column with the respective original values
    df_part[translated_col] = df_part.apply(
    lambda row: row[oe_col] if pd.isna(row[translated_col]) or str(row[translated_col]).strip() == '' else row[translated_col],
    axis=1
)

    
    
    





In [ ]:
#     # Remove rows where the current oe_col is blank or only whitespace
#     df_part = df_part[df_part[oe_col].astype(str).str.strip() != '']
#     # Add an index column named "Res_ID" as the first column
#     df_part.insert(0, 'Res_ID', range(1, len(df_part) + 1))

#     # Translate the 'oe_col' column to English
#     translated_col = f"{oe_col}_Trans"

#     if EU_client:
#         df_part[translated_col] = df_part[oe_col].apply(lambda x: deepl_client.translate_text(str(x), target_lang="EN-US").text)

#     else:
#         df_part[translated_col] = df_part[oe_col].apply(lambda x: translator.translate(str(x)))
#     # Move the translated column to the third position
#     df_part.insert(2, translated_col, df_part.pop(translated_col))

#     # replace all the blank values in the translated column with the respective original values
#     df_part[translated_col] = df_part.apply(
#     lambda row: row[oe_col] if pd.isna(row[translated_col]) or str(row[translated_col]).strip() == '' else row[translated_col],
#     axis=1)
    
#     # Sort by length of translated text (descending)
#     df_part = df_part.sort_values(by=translated_col, key=lambda col: col.str.len(), ascending=True)
#     # Save to dictionary
#     oe_dfs[oe_col] = df_part

#     translator_name = "DeepL" if EU_client else "Google"
#     output_path = os.path.join(output_folder, f"{company_name}_{oe_col}_{translator_name}_Trans.xlsx" # type: ignore
# )
#     df_part.to_excel(output_path, index=False)
